# 🔬 EDA — NASA IMS Bearing Dataset
### Agentic AI-Based Industrial Equipment Health Monitoring & Fault Diagnosis

---

**Dataset:** NASA IMS Bearing Dataset (Center for Intelligent Maintenance Systems, University of Cincinnati)  
**Kaggle:** https://www.kaggle.com/datasets/vinayak123tyagi/bearing-dataset  
**License:** U.S. Government Works  

---

## 📋 Table of Contents
1. Dataset Overview & Context
2. Environment Setup & Imports
3. Data Loading (Real + DEMO mode)
4. File-Level Statistics
5. Raw Signal Visualization
6. Time-Domain Feature Engineering
7. Degradation Trend Analysis
8. Feature Dashboard (8 features)
9. Statistical Distribution Analysis
10. Frequency-Domain Analysis (FFT)
11. Spectrum Evolution (Pseudo-Spectrogram)
12. Feature Correlation Analysis
13. Cross-Bearing RMS Comparison
14. Anomaly Region Identification (Z-Score)
15. Health Index Construction (Baseline)
16. Feature Discriminative Power
17. Key EDA Findings & Next Steps

---

## ⚠️ Dataset Setup Instructions

**Before running this notebook**, download from Kaggle and extract to:
```
data/raw/bearing_dataset/
├── 1st_test/   ← Set 1 (8 channels, Oct–Nov 2003, 2,156 files)
├── 2nd_test/   ← Set 2 (4 channels, Feb 2004, 984 files)
└── 4th_test/   ← Set 3 (4 channels, Mar–Apr 2004, 4,448 files)
```
If the dataset is not present, the notebook automatically enters **DEMO MODE**
with realistic synthetic data to demonstrate the full analysis workflow.

---
## 1. Dataset Overview & Context

### Experimental Setup
Four **Rexnord ZA-2115** double-row bearings on a single shaft:
- Shaft speed: **2,000 RPM** (constant)
- Radial load: **6,000 lbs** (spring mechanism)
- Force-lubricated
- All failures exceeded designed bearing life (>100 million revolutions)

### Dataset Summary

| Set | Date Range | Files | Channels | Failed Bearing | Failure Type |
|-----|-----------|-------|----------|----------------|---------------|
| Set 1 | Oct 22–Nov 25, 2003 | 2,156 | 8 (x+y per bearing) | B3, B4 | Inner Race + Rolling Element |
| Set 2 | Feb 12–19, 2004 | 984 | 4 (1 per bearing) | B1 | Outer Race |
| Set 3 | Mar 4–Apr 4, 2004 | 4,448 | 4 (1 per bearing) | B3 | Outer Race |

**Signal:** 20,480 samples per 1-second snapshot @ 20 kHz, every 10 minutes.

### Rexnord ZA-2115 Bearing Fault Frequencies @ 2000 RPM

| Fault Type | Abbrev | Frequency (Hz) | Associated Failure |
|------------|--------|----------------|--------------------|
| Ball Pass Freq – Outer | BPFO | 236.4 | Set2-B1, Set3-B3 |
| Ball Pass Freq – Inner | BPFI | 296.9 | Set1-B3 |
| Ball Spin Frequency | BSF | 139.9 | Set1-B4 |
| Fundamental Train Freq | FTF | 14.7 | Cage |

> **Note:** These are theoretical values. Peaks may shift slightly due to bearing slip.

---
## 2. Environment Setup & Imports

In [ ]:
import os
import warnings
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ─── Dark theme ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d0d1a',
    'axes.facecolor':   '#16162a',
    'axes.edgecolor':   '#3a3a5c',
    'axes.labelcolor':  '#c0c0e0',
    'axes.titlecolor':  '#e8e8ff',
    'xtick.color':      '#9090b0',
    'ytick.color':      '#9090b0',
    'text.color':       '#e0e0ff',
    'grid.color':       '#252540',
    'grid.linestyle':   '--',
    'grid.alpha':       0.7,
    'lines.linewidth':  1.2,
    'legend.facecolor': '#16162a',
    'legend.edgecolor': '#3a3a5c',
    'legend.labelcolor': '#c0c0e0',
    'figure.titlesize': 15,
    'axes.titlesize':   12,
    'axes.labelsize':   10,
})

# ─── Color palette ─────────────────────────────────────────────────────────
C = {
    'b1': '#00d4ff', 'b2': '#7b68ee',
    'b3': '#ff6b6b', 'b4': '#ffd93d',
    'ok': '#00ff9f', 'warn': '#ffa500',
    'bad': '#ff3333', 'acc': '#a855f7',
}
PALETTE = [C['b1'], C['b2'], C['b3'], C['b4'], C['acc'], C['warn']]
FAULT_COLORS = {'BPFO': '#ffd700', 'BPFI': '#ff8c00', 'BSF': '#9370db', 'FTF': '#20b2aa'}

print(f'NumPy    {np.__version__}')
print(f'Pandas   {pd.__version__}')
print(f'Matplotlib {matplotlib.__version__}')
print(f'Seaborn  {sns.__version__}')
print('✅ Environment ready')

---
## 3. Data Loading

In [ ]:
# ─── Constants ────────────────────────────────────────────────────────────
SAMPLING_RATE      = 20_000   # Hz
N_SAMPLES_PER_FILE = 20_480   # points per 1-second snapshot
SHAFT_RPM          = 2_000
SHAFT_HZ           = SHAFT_RPM / 60

FAULT_FREQ = {
    'BPFO': 236.4,  # Ball Pass Freq Outer race
    'BPFI': 296.9,  # Ball Pass Freq Inner race
    'BSF':  139.9,  # Ball Spin Frequency
    'FTF':  14.7,   # Fundamental Train Frequency
}

# ─── Set metadata from dataset documentation ───────────────────────────────
SET_META = {
    'Set1': {
        'n_ch': 8,
        'bearings': ['B1_x', 'B1_y', 'B2_x', 'B2_y', 'B3_x', 'B3_y', 'B4_x', 'B4_y'],
        'failed': ['B3', 'B4'],
        'fault_modes': {'B3': 'Inner Race', 'B4': 'Rolling Element'},
    },
    'Set2': {
        'n_ch': 4,
        'bearings': ['B1', 'B2', 'B3', 'B4'],
        'failed': ['B1'],
        'fault_modes': {'B1': 'Outer Race'},
    },
    'Set3': {
        'n_ch': 4,
        'bearings': ['B1', 'B2', 'B3', 'B4'],
        'failed': ['B3'],
        'fault_modes': {'B3': 'Outer Race'},
    },
}

# ─── Paths ────────────────────────────────────────────────────────────────
NB_DIR       = Path().resolve()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
DATA_ROOT    = PROJECT_ROOT / 'data' / 'raw' / 'bearing_dataset'

print(f'Project root : {PROJECT_ROOT}')
print(f'Data root    : {DATA_ROOT}')
print(f'Data exists  : {DATA_ROOT.exists()}')

# ─── Auto-detect dataset folders ──────────────────────────────────────────
FOLDER_MAP = {}
if DATA_ROOT.exists():
    subdirs = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
    print(f'Subdirs found: {subdirs}')
    for i, cand in enumerate(['1st_test', '2nd_test', '4th_test'], 1):
        if cand in subdirs:
            FOLDER_MAP[f'Set{i}'] = DATA_ROOT / cand
    if not FOLDER_MAP:
        for i, d in enumerate(subdirs[:3], 1):
            FOLDER_MAP[f'Set{i}'] = DATA_ROOT / d

DEMO_MODE = len(FOLDER_MAP) == 0
print(f'\nDemo Mode: {DEMO_MODE}')
if not DEMO_MODE:
    for k, v in FOLDER_MAP.items():
        n = len(list(v.glob('*')))
        print(f'  {k}: {v.name}  ({n} files)')

In [ ]:
def load_bearing_file(fp: Path) -> np.ndarray:
    """Load one IMS ASCII data file. Returns (n_samples, n_channels)."""
    d = np.loadtxt(fp)
    return d.reshape(-1, 1) if d.ndim == 1 else d


def make_synthetic(n_files: int = 150, n_ch: int = 4, seed: int = 42):
    """
    Simulate run-to-failure bearing vibration.
    Channel 0 is the faulty bearing with growing BPFO impulses.
    """
    np.random.seed(seed % (2**31))
    t = np.linspace(0, 1, N_SAMPLES_PER_FILE)
    signals = {}
    for ch in range(n_ch):
        arr = []
        for i in range(n_files):
            deg  = (i / n_files) ** 2         # degradation grows nonlinearly
            base = (0.1 * np.sin(2 * np.pi * SHAFT_HZ * t)
                    + sum(0.03 / k * np.sin(2 * np.pi * k * SHAFT_HZ * t)
                          for k in range(2, 5))
                    + 0.02 * np.random.randn(N_SAMPLES_PER_FILE))
            # Fault impulses at BPFO rate, amplitude grows with degradation
            amp  = deg * (0.5 if ch == 0 else 0.05)
            period = int(SAMPLING_RATE / FAULT_FREQ['BPFO'])
            imp  = np.zeros(N_SAMPLES_PER_FILE)
            for k in range(0, N_SAMPLES_PER_FILE, period):
                w = min(180, N_SAMPLES_PER_FILE - k)
                imp[k:k + w] += amp * np.exp(-40 * np.linspace(0, 1, w))
            arr.append(base + imp)
        signals[f'B{ch + 1}'] = np.array(arr)  # (n_files, N_SAMPLES_PER_FILE)
    timestamps = [datetime(2024, 1, 1) + pd.Timedelta(minutes=10 * i)
                  for i in range(n_files)]
    meta = pd.DataFrame({'file_index': range(n_files), 'timestamp': timestamps})
    return signals, meta


# ─── Load or generate data ─────────────────────────────────────────────────
ALL_DATA = {}   # {set_name: {'signals': {bname: arr}, 'meta': df}}

if DEMO_MODE:
    print('⚡ DEMO MODE — generating synthetic bearing degradation data')
    for sname, smeta in SET_META.items():
        n_f = {'Set1': 200, 'Set2': 150, 'Set3': 180}[sname]
        seed = {'Set1': 42, 'Set2': 99, 'Set3': 7}[sname]
        sig, meta = make_synthetic(n_files=n_f, n_ch=smeta['n_ch'], seed=seed)
        # Rename channels to documented bearing names
        bnames = smeta['bearings']
        sig = {bnames[k]: v for k, v in enumerate(sig.values())
               if k < len(bnames)}
        ALL_DATA[sname] = {'signals': sig, 'meta': meta}
        print(f'  {sname}: {n_f} snapshots x {smeta["n_ch"]} channels (synthetic)')
else:
    print('📂 Loading REAL dataset...')
    for sname, folder in FOLDER_MAP.items():
        smeta_info = SET_META[sname]
        all_files = sorted([f for f in folder.glob('*') if f.is_file()])
        # Sample at most 250 files for EDA
        step = max(1, len(all_files) // 250)
        sampled = all_files[::step]
        stacked = []
        for fp in sampled:
            try:
                d = load_bearing_file(fp)
                stacked.append(d[:N_SAMPLES_PER_FILE, :smeta_info['n_ch']])
            except Exception as ex:
                logger.warning(f'Skip {fp.name}: {ex}')
        if stacked:
            arr3d = np.stack(stacked)  # (n_files, n_samples, n_ch)
            bnames = smeta_info['bearings']
            sig = {bnames[c]: arr3d[:, :, c] for c in range(min(smeta_info['n_ch'], arr3d.shape[2]))}
            ts = [datetime(2020, 1, 1) + pd.Timedelta(minutes=10 * i)
                  for i in range(len(stacked))]
            meta = pd.DataFrame({'file_index': range(len(stacked)), 'timestamp': ts})
            ALL_DATA[sname] = {'signals': sig, 'meta': meta}
            print(f'  {sname}: {len(stacked)} snapshots loaded')

print(f'\n✅ Datasets ready: {list(ALL_DATA.keys())}')

---
## 4. File-Level Statistics

In [ ]:
rows = []
for sname, ds in ALL_DATA.items():
    sm = SET_META[sname]
    n_f = ds['meta'].shape[0]
    rows.append({
        'Set': sname, 'Snapshots': n_f, 'Channels': sm['n_ch'],
        'Duration_hrs': round(n_f * 10 / 60, 1),
        'Failed': ', '.join(sm['failed']),
        'Fault_Mode': ', '.join(sm['fault_modes'].values()),
        'Mode': 'DEMO' if DEMO_MODE else 'REAL',
    })
print('\nDataset Summary:')
print(pd.DataFrame(rows).to_string(index=False))

print()
for sname, ds in ALL_DATA.items():
    print(f'\n{sname} — Channel Stats (snapshot 0 = initial healthy state):')
    hdr = f'{"Channel":<12}{"Mean":>10}{"Std":>10}{"RMS":>10}{"Kurtosis":>12}{"Peak":>10}'
    print(hdr)
    print('-' * len(hdr))
    for bname, sarr in ds['signals'].items():
        s = sarr[0].astype(float)
        print(f'{bname:<12}{np.mean(s):>10.5f}{np.std(s):>10.5f}'
              f'{np.sqrt(np.mean(s**2)):>10.5f}{kurtosis(s):>12.3f}'
              f'{np.max(np.abs(s)):>10.5f}')

---
## 5. Raw Signal Visualization
Three snapshots per bearing: **Early (healthy)**, **Mid**, **Late (near failure)**.

In [ ]:
def plot_raw_signals(ds, sname, n_plot=2048):
    sigs  = ds['signals']
    blist = list(sigs.keys())
    sm    = SET_META[sname]
    n_b   = len(blist)
    n_f   = list(sigs.values())[0].shape[0]
    idxs  = [0, n_f // 2, max(n_f - 5, 0)]
    lbls  = ['Early (Healthy)', 'Mid', 'Late (Degraded)']
    t_ms  = np.linspace(0, n_plot / SAMPLING_RATE * 1000, n_plot)

    fig, axes = plt.subplots(n_b, 3, figsize=(15, 2.5 * n_b), squeeze=False)
    fig.suptitle(f'{sname} — Raw Vibration Waveforms (first {n_plot} samples)',
                 fontweight='bold')
    for bi, bn in enumerate(blist):
        base   = bn.split('_')[0]
        failed = base in sm['failed']
        col    = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
        for si, (fi, lbl) in enumerate(zip(idxs, lbls)):
            ax  = axes[bi][si]
            s   = sigs[bn][fi, :n_plot]
            rms = float(np.sqrt(np.mean(s ** 2)))
            ax.plot(t_ms, s, color=col, lw=0.7)
            ax.axhline(0, color='#44446a', lw=0.5)
            ax.text(0.97, 0.94, f'RMS={rms:.4f}', transform=ax.transAxes,
                    fontsize=7, ha='right', va='top', color='#ffffaa',
                    bbox=dict(boxstyle='round,pad=0.2', fc='#111133', alpha=0.85))
            ax.grid(True)
            if bi == 0:
                ax.set_title(lbl, fontsize=9)
            if si == 0:
                ax.set_ylabel(f'{bn}{" ⚠️" if failed else ""}', fontsize=8)
            if bi == n_b - 1:
                ax.set_xlabel('Time (ms)', fontsize=8)
    plt.tight_layout()
    plt.show()

for sname, ds in ALL_DATA.items():
    plot_raw_signals(ds, sname)

---
## 6. Time-Domain Feature Engineering
| Feature | Formula | Fault Sensitivity |
|---------|---------|-------------------|
| **RMS** | √(mean(x²)) | Overall energy — increases monotonically |
| **Kurtosis** | 4th moment | Early impulsive faults (kurtosis paradox at late stage) |
| **Crest Factor** | Peak/RMS | Impulse severity |
| **Skewness** | 3rd moment | Distribution asymmetry |
| **Peak-to-Peak** | max(x)−min(x) | Raw amplitude range |
| **Shape Factor** | RMS/mean(|x|) | Waveform shape changes |
| **Impulse Factor** | Peak/mean(|x|) | Impulsive content |
| **Clearance Factor** | Peak/(√mean(|x|))² | Lubrication degradation |

In [ ]:
def td_features(s: np.ndarray) -> dict:
    """Compute time-domain vibration analysis features for one signal snapshot."""
    s   = s.astype(np.float64)
    a   = np.abs(s)
    rms = float(np.sqrt(np.mean(s ** 2)))
    pk  = float(np.max(a))
    ma  = float(np.mean(a)) + 1e-12
    sm_ = float(np.mean(np.sqrt(a))) + 1e-12
    return {
        'rms':              rms,
        'peak':             pk,
        'mean_abs':         ma,
        'variance':         float(np.var(s)),
        'kurtosis':         float(kurtosis(s)),
        'skewness':         float(skew(s)),
        'peak2peak':        float(np.max(s) - np.min(s)),
        'crest_factor':     pk / (rms + 1e-12),
        'shape_factor':     rms / ma,
        'impulse_factor':   pk / ma,
        'clearance_factor': pk / (sm_ ** 2),
    }


def build_features(ds: dict, sname: str) -> pd.DataFrame:
    rows = []
    for bname, sarr in ds['signals'].items():
        for i in range(len(sarr)):
            r = {'set': sname, 'bearing': bname, 'file_index': i}
            r.update(td_features(sarr[i]))
            rows.append(r)
    return pd.DataFrame(rows)


ALL_FEATS = {}
for sname, ds in ALL_DATA.items():
    print(f'Extracting features: {sname}...')
    df = build_features(ds, sname)
    ALL_FEATS[sname] = df
    print(f'  → shape: {df.shape}')

# Sample output
if 'Set2' in ALL_FEATS:
    sample = ALL_FEATS['Set2'].head(1)
    fcols = ['rms', 'kurtosis', 'crest_factor', 'skewness', 'peak2peak']
    print('\nSample (Set2, file 0):')
    print(sample[fcols].to_string(index=False))

print('\n✅ Feature extraction complete')

---
## 7. Degradation Trend Analysis
These plots reveal the **run-to-failure degradation progression** for each feature.
Failed bearings are highlighted in red. The failure region (~last 15%) is shaded.

In [ ]:
def plot_trends(feat_df: pd.DataFrame, sname: str, features: list):
    sm      = SET_META[sname]
    bearings = feat_df['bearing'].unique()
    n_f     = len(features)
    fig, axes = plt.subplots(n_f, 1, figsize=(14, 3.8 * n_f),
                              sharex=True, squeeze=False)
    fig.suptitle(f'{sname} — Feature Degradation Trends (Run-to-Failure)',
                 fontweight='bold')
    for fi, feat in enumerate(features):
        ax = axes[fi][0]
        for bi, bn in enumerate(bearings):
            sub = feat_df[feat_df['bearing'] == bn].sort_values('file_index')
            x, y  = sub['file_index'].values, sub[feat].values
            base  = bn.split('_')[0]
            failed = base in sm['failed']
            col   = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
            lbl   = (f'{bn} ⚠️ [{sm["fault_modes"].get(base, "")}]'
                     if failed else bn)
            ax.plot(x, y, color=col, lw=1.8 if failed else 1.0,
                    label=lbl, zorder=5 if failed else 3, alpha=0.9)
            if failed:
                fs = int(len(x) * 0.85)
                ax.axvspan(x[fs], x[-1], color=col, alpha=0.07, zorder=1)
                ax.axvline(x[fs], color=col, ls='--', lw=0.9, alpha=0.5)
        ax.set_ylabel(feat.replace('_', ' ').upper(), fontsize=9)
        ax.grid(True)
        ax.legend(fontsize=8, ncol=2, loc='upper left')
    axes[-1][0].set_xlabel('Snapshot Index (time →)', fontsize=9)
    plt.tight_layout()
    plt.show()


for sname, feat_df in ALL_FEATS.items():
    plot_trends(feat_df, sname,
                features=['rms', 'kurtosis', 'crest_factor', 'peak2peak'])

---
## 8. Feature Dashboard (8 Features)

In [ ]:
def feature_dashboard(feat_df: pd.DataFrame, sname: str):
    feats    = ['rms', 'kurtosis', 'crest_factor', 'skewness',
                'peak2peak', 'impulse_factor', 'clearance_factor', 'variance']
    sm       = SET_META[sname]
    bearings = feat_df['bearing'].unique()
    fig, axes = plt.subplots(4, 2, figsize=(16, 3.6 * 4), squeeze=False)
    fig.suptitle(f'{sname} — 8-Feature Degradation Dashboard', fontweight='bold', y=1.01)
    for fi, feat in enumerate(feats):
        ax = axes[fi // 2][fi % 2]
        for bi, bn in enumerate(bearings):
            sub = feat_df[feat_df['bearing'] == bn].sort_values('file_index')
            x, y  = sub['file_index'].values, sub[feat].values
            base  = bn.split('_')[0]
            failed = base in sm['failed']
            col   = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
            ax.plot(x, y, color=col, lw=1.8 if failed else 0.9,
                    label=f'{bn}⚠️' if failed else bn, alpha=0.88)
        ax.set_title(feat.replace('_', ' ').upper(), fontsize=10, pad=3)
        ax.set_xlabel('Snapshot', fontsize=8)
        ax.grid(True)
        ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()


for sname, feat_df in ALL_FEATS.items():
    feature_dashboard(feat_df, sname)

---
## 9. Statistical Distribution Analysis
Amplitude histograms comparing **early (healthy)** vs **late (degraded)** phases.
Key observations:
- Healthy signals: near-Gaussian, low kurtosis
- Degraded signals: heavy tails, high kurtosis, wider spread

In [ ]:
def plot_distributions(ds, sname, n_early=10, n_late=10):
    sigs   = ds['signals']
    sm     = SET_META[sname]
    bnames = list(sigs.keys())
    n_b    = len(bnames)
    fig, axes = plt.subplots(2, n_b, figsize=(4 * n_b, 8), squeeze=False)
    fig.suptitle(f'{sname} — Amplitude Distribution: Early vs Late Phase',
                 fontweight='bold')
    for bi, bn in enumerate(bnames):
        arr    = sigs[bn]
        n_f    = len(arr)
        early  = arr[:n_early].flatten()
        late   = arr[max(0, n_f - n_late):].flatten()
        base   = bn.split('_')[0]
        failed = base in sm['failed']
        bc     = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
        for pi, (data, lbl, col) in enumerate([
            (early, 'Early (Healthy)', C['ok']),
            (late,  'Late (Degraded)', bc),
        ]):
            ax  = axes[pi][bi]
            sns.histplot(data, bins=80, kde=True, ax=ax, color=col,
                         alpha=0.6, stat='density', line_kws={'lw': 2})
            rms = float(np.sqrt(np.mean(data ** 2)))
            k   = float(kurtosis(data))
            ax.set_title(f'{bn}\n{lbl}', fontsize=9)
            ax.set_xlabel('Amplitude (g)', fontsize=8)
            ax.set_ylabel('Density', fontsize=8)
            ax.text(0.97, 0.93, f'RMS={rms:.4f}\nKurt={k:.2f}',
                    transform=ax.transAxes, fontsize=8, ha='right', va='top',
                    color='#ffffaa',
                    bbox=dict(boxstyle='round,pad=0.3', fc='#0a0a22', alpha=0.85))
            ax.grid(True)
            if failed and pi == 1:
                ax.set_facecolor('#1a0808')
    plt.tight_layout()
    plt.show()


for sname, ds in ALL_DATA.items():
    plot_distributions(ds, sname)

In [ ]:
def plot_boxplots(feat_df: pd.DataFrame, sname: str):
    sm       = SET_META[sname]
    bearings = feat_df['bearing'].unique()
    feats    = ['rms', 'kurtosis', 'crest_factor', 'peak2peak']
    pal      = {bn: (C['bad'] if bn.split('_')[0] in sm['failed']
                     else PALETTE[i % len(PALETTE)])
                for i, bn in enumerate(bearings)}
    fig, axes = plt.subplots(1, 4, figsize=(5 * 4, 5))
    fig.suptitle(f'{sname} — Feature Distribution by Bearing (Box Plots)',
                 fontweight='bold')
    for ax, feat in zip(axes, feats):
        sns.boxplot(data=feat_df, x='bearing', y=feat, ax=ax,
                    palette=pal, linewidth=0.9,
                    flierprops=dict(marker='o', markersize=2, alpha=0.3))
        ax.set_title(feat.replace('_', ' ').upper(), fontsize=10)
        ax.set_xlabel('Bearing', fontsize=9)
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        ax.grid(True, axis='y')
    plt.tight_layout()
    plt.show()


for sname, feat_df in ALL_FEATS.items():
    plot_boxplots(feat_df, sname)

---
## 10. Frequency-Domain Analysis (FFT)
Bearing fault signatures manifest as **periodic impulses** at characteristic frequencies.
The one-sided FFT (with Hanning window to reduce spectral leakage) reveals these.

- Peaks at BPFO/BPFI/BSF indicate specific fault types
- Harmonics indicate progressing fault severity  
- High-frequency energy growth indicates surface/lubrication damage
- Sidebands around shaft harmonics indicate amplitude modulation

In [ ]:
def compute_fft(sig: np.ndarray, fs: int = SAMPLING_RATE):
    """One-sided FFT with Hanning window. Returns (freqs, amplitude)."""
    n     = len(sig)
    w     = np.hanning(n)
    mag   = np.abs(np.fft.fft(sig * w))[:n // 2] * 2 / n
    freqs = np.fft.fftfreq(n, 1 / fs)[:n // 2]
    return freqs, mag


def plot_fft(ds, sname, freq_max=5000):
    sigs  = ds['signals']
    sm    = SET_META[sname]
    blist = list(sigs.keys())
    n_b   = len(blist)
    fig, axes = plt.subplots(n_b, 2, figsize=(16, 3.2 * n_b), squeeze=False)
    fig.suptitle(f'{sname} — FFT Spectrum: Early vs Late Phase',
                 fontweight='bold', y=1.01)
    for bi, bn in enumerate(blist):
        arr    = sigs[bn]
        n_f    = len(arr)
        base   = bn.split('_')[0]
        failed = base in sm['failed']
        bc     = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
        for pi, (fi, lbl, col) in enumerate([
            (0,       'Early (Healthy)', C['ok']),
            (n_f - 1, 'Late (Degraded)', bc),
        ]):
            ax         = axes[bi][pi]
            freqs, mag = compute_fft(arr[fi])
            mask       = freqs <= freq_max
            ax.plot(freqs[mask], mag[mask], color=col, lw=0.7, alpha=0.9)
            ax.fill_between(freqs[mask], 0, mag[mask], color=col, alpha=0.12)
            for fname, fhz in FAULT_FREQ.items():
                if fhz <= freq_max:
                    ax.axvline(fhz, color=FAULT_COLORS[fname], ls='--',
                               lw=0.9, alpha=0.75,
                               label=f'{fname}={fhz}Hz' if bi == 0 else '')
                    if 2 * fhz <= freq_max:
                        ax.axvline(2 * fhz, color=FAULT_COLORS[fname],
                                   ls=':', lw=0.6, alpha=0.4)
            title_suf = '⚠️ FAILED' if failed else ''
            ax.set_title(f'{bn} {title_suf} — {lbl}', fontsize=9)
            ax.set_xlabel('Frequency (Hz)', fontsize=8)
            ax.set_ylabel('Amplitude', fontsize=8)
            ax.set_xlim(0, freq_max)
            ax.grid(True)
            if failed and pi == 1:
                ax.set_facecolor('#1a0808')
            if bi == 0 and pi == 0:
                ax.legend(fontsize=7, ncol=2, loc='upper right')
    plt.tight_layout()
    plt.show()


for sname, ds in ALL_DATA.items():
    plot_fft(ds, sname)

---
## 11. Spectrum Evolution (Pseudo-Spectrogram)
Stacking FFT spectra over the run-to-failure timeline shows **when** specific frequency
components emerge and intensify. This is the time-frequency view of degradation.

In [ ]:
def plot_spectrogram(ds, sname, bn=None, n_use=60, freq_max=5000):
    sigs = ds['signals']
    sm   = SET_META[sname]
    # Auto-pick: prefer known failed bearing
    if bn is None:
        for b in sigs:
            if b.split('_')[0] in sm['failed']:
                bn = b
                break
        if bn is None:
            bn = list(sigs.keys())[0]
    arr    = sigs[bn]
    n_f    = len(arr)
    step   = max(1, n_f // n_use)
    idxs   = list(range(0, n_f, step))[:n_use]
    spectra, freq_ax = [], None
    for idx in idxs:
        freqs, mag = compute_fft(arr[idx])
        mask = freqs <= freq_max
        if freq_ax is None:
            freq_ax = freqs[mask]
        spectra.append(mag[mask])
    S      = np.array(spectra)
    failed = bn.split('_')[0] in sm['failed']
    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.pcolormesh(freq_ax, np.array(idxs), S,
                       cmap='inferno', shading='auto',
                       vmin=0, vmax=np.percentile(S, 99))
    plt.colorbar(im, ax=ax, label='Amplitude', shrink=0.8)
    for fname, fhz in FAULT_FREQ.items():
        if fhz <= freq_max:
            ax.axvline(fhz, color='#00ffff', ls='--', lw=0.8,
                       alpha=0.7, label=f'{fname}={fhz}Hz')
    fail_mark = '⚠️ [FAILED]' if failed else ''
    ax.set_title(f'{sname} — {bn} {fail_mark}\nSpectrum Evolution Over Run-to-Failure',
                 fontweight='bold')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Snapshot Index (time →)')
    ax.legend(fontsize=8, ncol=2, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'  {sname} | {bn} | Failed: {failed} | {len(idxs)}/{n_f} snapshots')


for sname, ds in ALL_DATA.items():
    plot_spectrogram(ds, sname)

---
## 12. Feature Correlation Analysis
Identifies **redundant** and **independent** features for model design.
High correlation (|r|>0.85) → consider using only one of the pair to avoid redundancy.

In [ ]:
def plot_correlation(feat_df: pd.DataFrame, sname: str):
    fcols = ['rms', 'kurtosis', 'crest_factor', 'skewness',
             'peak2peak', 'impulse_factor', 'clearance_factor',
             'variance', 'mean_abs']
    fcols = [c for c in fcols if c in feat_df.columns]
    corr  = feat_df[fcols].corr()
    mask  = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True,
                linewidths=0.4, ax=ax,
                cbar_kws={'shrink': 0.8},
                annot_kws={'size': 8}, vmin=-1, vmax=1)
    ax.set_title(f'{sname} — Feature Correlation Matrix (All Bearings)',
                 fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()
    high = [(c1, c2, corr.loc[c1, c2])
            for i, c1 in enumerate(fcols)
            for j, c2 in enumerate(fcols)
            if i < j and abs(corr.loc[c1, c2]) > 0.85]
    if high:
        print(f'Highly correlated pairs in {sname} (|r|>0.85):')
        for c1, c2, r in high:
            print(f'  {c1} ↔ {c2}: r={r:.3f}')
    else:
        print(f'No pairs with |r|>0.85 in {sname}')


for sname, feat_df in ALL_FEATS.items():
    plot_correlation(feat_df, sname)

---
## 13. Cross-Bearing RMS Comparison

In [ ]:
def plot_rms_comparison(feat_df: pd.DataFrame, sname: str):
    sm       = SET_META[sname]
    bearings = feat_df['bearing'].unique()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'{sname} — Cross-Bearing RMS Comparison', fontweight='bold')
    rms_dict = {}
    for bi, bn in enumerate(bearings):
        sub = feat_df[feat_df['bearing'] == bn].sort_values('file_index')
        rms_dict[bn] = sub['rms'].values
    for bi, (bn, vals) in enumerate(rms_dict.items()):
        base   = bn.split('_')[0]
        failed = base in sm['failed']
        col    = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
        lbl    = f'{bn} ⚠️' if failed else bn
        ax1.plot(vals, color=col, lw=1.8 if failed else 1.1, label=lbl, alpha=0.9)
        init = vals[0] if vals[0] > 1e-10 else 1.0
        ax2.plot(vals / init, color=col, lw=1.8 if failed else 1.1, label=lbl, alpha=0.9)
    ax1.set_title('Absolute RMS (g)', fontsize=11)
    ax1.set_xlabel('Snapshot Index')
    ax1.set_ylabel('RMS (g)')
    ax1.grid(True)
    ax1.legend(fontsize=8)
    ax2.set_title('Normalized RMS (/ Initial Value)', fontsize=11)
    ax2.set_xlabel('Snapshot Index')
    ax2.set_ylabel('RMS / RMS₀')
    ax2.axhline(1.0, color='#888', ls='--', lw=0.8, label='Baseline')
    ax2.grid(True)
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


for sname, feat_df in ALL_FEATS.items():
    plot_rms_comparison(feat_df, sname)

---
## 14. Anomaly Region Identification (Z-Score)
**Method:** Compute mean and std from the first N snapshots (healthy baseline).
Flag any snapshot where the absolute z-score exceeds a threshold.

This is a **simple statistical baseline**. More rigorous ML-based anomaly detection
(Isolation Forest, LSTM Autoencoder) will follow in **Phase 6**.

In [ ]:
def zscore_anomaly(series: pd.Series, n_baseline: int = 10,
                    threshold: float = 3.0) -> np.ndarray:
    """Z-score anomaly detection using initial healthy baseline statistics."""
    v  = series.values.astype(float)
    mu = np.mean(v[:n_baseline])
    sd = np.std(v[:n_baseline])
    if sd < 1e-10:
        return np.zeros(len(v), dtype=bool)
    return np.abs((v - mu) / sd) > threshold


def plot_anomalies(feat_df: pd.DataFrame, sname: str,
                   feat: str = 'rms', thresh: float = 3.0):
    sm       = SET_META[sname]
    bearings = feat_df['bearing'].unique()
    fig, axes = plt.subplots(len(bearings), 1,
                              figsize=(14, 3.2 * len(bearings)), squeeze=False)
    fig.suptitle(
        f'{sname} — Z-Score Anomaly Detection (|z|>{thresh}) — {feat.upper()}',
        fontweight='bold')
    summary = []
    for bi, bn in enumerate(bearings):
        sub    = feat_df[feat_df['bearing'] == bn].sort_values('file_index')
        x, y   = sub['file_index'].values, sub[feat].values
        n_bl   = max(5, len(sub) // 10)
        anom   = zscore_anomaly(sub[feat], n_bl, thresh)
        n_anom = int(anom.sum())
        base   = bn.split('_')[0]
        failed = base in sm['failed']
        col    = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
        ax     = axes[bi][0]
        ax.plot(x, y, color=col, lw=1.2, alpha=0.8, label=bn)
        if n_anom:
            ax.scatter(x[anom], y[anom], c='#ff0000', s=20,
                       zorder=5, alpha=0.7, label=f'Anomaly ({n_anom})')
            ax.axvline(x[anom][0], color='#ff6600', ls='--',
                       lw=0.9, alpha=0.8, label=f'1st@{x[anom][0]}')
        suf = f' ⚠️[{sm["fault_modes"].get(base, "")}]' if failed else ''
        ax.set_title(f'{bn}{suf}  |  Anomalies: {n_anom}/{len(x)}', fontsize=10)
        ax.set_xlabel('Snapshot Index')
        ax.set_ylabel(feat.upper(), fontsize=9)
        ax.grid(True)
        ax.legend(fontsize=8)
        summary.append({'bearing': bn, 'n_anom': n_anom,
                         'rate_%': round(100 * n_anom / len(x), 1),
                         'failed': failed})
    plt.tight_layout()
    plt.show()
    print(f'\nAnomaly Summary — {sname}:')
    print(pd.DataFrame(summary).to_string(index=False))


for sname, feat_df in ALL_FEATS.items():
    plot_anomalies(feat_df, sname)

---
## 15. Health Index Construction (Baseline)
A **Health Index (HI)** converts multi-feature degradation into a single **[0–100]** score:
- **100** = Perfectly healthy (baseline state)
- **0** = Complete failure

**Method:** Weighted exponential degradation model using 4 key features.

This provides intuitive, explainable health monitoring. A more accurate ML-based
health model will be implemented in **Phase 7** of the project.

In [ ]:
# Feature weights for health index computation
HI_WEIGHTS = {
    'rms':          0.35,
    'kurtosis':     0.25,
    'crest_factor': 0.20,
    'peak2peak':    0.20,
}


def health_index(feat_df: pd.DataFrame, bn: str, n_bl: int = 10) -> np.ndarray:
    """
    Compute multi-feature weighted health index for one bearing.
    Normalizes features by their baseline (healthy) mean, then fuses.
    Returns array of shape (n_snapshots,) with values in [0, 100].
    """
    sub = feat_df[feat_df['bearing'] == bn].sort_values('file_index')
    deg = np.zeros(len(sub))
    for feat, w in HI_WEIGHTS.items():
        if feat not in sub.columns:
            continue
        v        = sub[feat].values.astype(float)
        baseline = np.mean(v[:n_bl]) + 1e-10
        deg     += w * (v / baseline)
    hi = np.clip(100.0 * np.exp(-(deg - 1.0)), 0, 100)
    return hi


def plot_health_index(feat_df: pd.DataFrame, sname: str):
    sm       = SET_META[sname]
    bearings = feat_df['bearing'].unique()
    fig, ax  = plt.subplots(figsize=(14, 6))
    fig.suptitle(f'{sname} — Bearing Health Index [0=Failed, 100=Healthy]',
                 fontweight='bold')
    # Health zone shading
    zones = [(75,100,'#00440050','Healthy (75-100)'),
             (50,75, '#44440050','Moderate (50-75)'),
             (25,50, '#44220050','Degraded (25-50)'),
             (0, 25, '#44000050','Critical (0-25)')]
    for lo, hi_, col, lbl in zones:
        ax.axhspan(lo, hi_, color=col, label=lbl)
    for th, col in [(75,'#00aa00'), (50,'#aaaa00'), (25,'#aa5500')]:
        ax.axhline(th, color=col, ls='--', lw=0.8, alpha=0.6)
    for bi, bn in enumerate(bearings):
        sub    = feat_df[feat_df['bearing'] == bn].sort_values('file_index')
        x      = sub['file_index'].values
        n_bl   = max(5, len(sub) // 10)
        hi_    = health_index(feat_df, bn, n_bl)
        base   = bn.split('_')[0]
        failed = base in sm['failed']
        col    = C['bad'] if failed else PALETTE[bi % len(PALETTE)]
        lbl    = (f'{bn} ⚠️ [{sm["fault_modes"].get(base, "")}]'
                  if failed else bn)
        ax.plot(x, hi_, color=col, lw=2.2 if failed else 1.3,
                label=lbl, alpha=0.92, zorder=5)
        ax.annotate(f'{hi_[-1]:.0f}',
                    xy=(x[-1], hi_[-1]),
                    xytext=(x[-1] + len(x) * 0.01, hi_[-1]),
                    fontsize=9, color=col, fontweight='bold')
    ax.set_xlabel('Snapshot Index (time →)', fontsize=10)
    ax.set_ylabel('Health Index [0-100]', fontsize=10)
    ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.4)
    ax.legend(fontsize=9, loc='lower left', ncol=2)
    plt.tight_layout()
    plt.show()
    # Print final HI values
    print(f'\n{sname} — Final Health Index Values:')
    for bi, bn in enumerate(bearings):
        hi_ = health_index(feat_df, bn)
        base = bn.split('_')[0]
        failed = base in sm['failed']
        status = '⚠️ FAILED' if failed else '✅ OK'
        print(f'  {bn:<12}: HI={hi_[-1]:5.1f}  {status}')


for sname, feat_df in ALL_FEATS.items():
    plot_health_index(feat_df, sname)

---
## 16. Feature Discriminative Power
Which features differ most between **failed** and **healthy** bearings?
A ratio >> 1.0 indicates the feature is significantly elevated in failed bearings.

In [ ]:
# Combine all sets
all_dfs = []
for sname, feat_df in ALL_FEATS.items():
    sm  = SET_META[sname]
    df  = feat_df.copy()
    df['is_failed'] = df['bearing'].apply(
        lambda b: b.split('_')[0] in sm['failed']
    )
    all_dfs.append(df)
combined = pd.concat(all_dfs, ignore_index=True)

FCOLS = ['rms', 'kurtosis', 'crest_factor', 'skewness',
         'peak2peak', 'impulse_factor', 'clearance_factor', 'variance']

grp   = combined.groupby('is_failed')[FCOLS].mean()
ratio = (grp.loc[True] / (grp.loc[False] + 1e-10)).sort_values(ascending=False)

print('Feature Discriminative Power (Failed / Healthy mean ratio):')
print(ratio.round(3).to_string())
print('\nInterpretation:')
print('  ratio >> 1.0  →  feature elevated in failed bearings (good discriminator)')
print('  ratio ≈ 1.0   →  feature similar in both conditions (low discriminator)')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [C['bad'] if v > 1.5 else C['b2'] for v in ratio.values]
ax.barh(ratio.index, ratio.values, color=bar_colors,
        edgecolor='#333355', lw=0.5)
ax.axvline(1.0, color='white', ls='--', lw=1.5, label='No difference (ratio=1)')
ax.set_xlabel('Failed / Healthy Mean Ratio', fontsize=10)
ax.set_title('Feature Discriminative Power: Failed vs Healthy Bearings',
             fontweight='bold')
ax.grid(True, axis='x', alpha=0.4)
ax.legend(fontsize=9)
for i, (feat, val) in enumerate(ratio.items()):
    ax.text(val + 0.02, i, f'{val:.2f}×', va='center',
            fontsize=9, color='#ffffaa')
plt.tight_layout()
plt.show()

---
## 17. Key EDA Findings & Next Steps

In [ ]:
print('=' * 68)
print('  EDA COMPLETE — NASA IMS BEARING DATASET')
print('=' * 68)
print(f'''
Mode: {"DEMO (Synthetic)" if DEMO_MODE else "REAL DATA"}

━━━ DATASET ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  3 run-to-failure experiments | 2,000 RPM | 6,000 lbs radial load
  20,480 samples/snapshot @ 20 kHz | Snapshot every 10 minutes
  Known failure modes: Inner Race, Outer Race, Rolling Element

━━━ TIME-DOMAIN FINDINGS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RMS          — primary, monotonic health indicator
  Kurtosis     — sensitive to early faults; kurtosis paradox at late stage
  Crest Factor — captures impulse severity growth
  Peak-to-Peak — raw amplitude growth near failure

━━━ FREQUENCY-DOMAIN FINDINGS ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BPFO (236 Hz)   → Outer race defects visible in late-stage FFT
  BPFI (297 Hz)   → Inner race defects
  High-freq energy growth → Surface/lubrication damage marker
  Spectrum evolution shows defect emergence timeline

━━━ FEATURE ENGINEERING ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RMS + Kurtosis + Crest Factor → best discriminative set
  RMS/Variance highly correlated (pick one per model)
  BPFO/BPFI band energy ratios → add to frequency feature set
  Multi-feature fusion → baseline health index functional

━━━ BASELINES DEMONSTRATED ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Health Index [0-100]: exponential degradation model
  Anomaly detection: z-score with healthy baseline window
  Both will be replaced by ML models in Phases 6 & 7

━━━ NEXT PHASES ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Phase 1 — Project Foundation         ✅ Ready
  Phase 2 — Dataset Pipeline           → data/loaders, preprocess, windowing
  Phase 3 — Signal Processing          ← This EDA notebook
  Phase 4 — Baseline Fault Diagnosis   → Random Forest, XGBoost
  Phase 5 — DL Fault Diagnosis         → 1D CNN
  Phase 6 — Anomaly Detection          → Isolation Forest, Autoencoder
  Phase 7 — Health Monitoring          → ML Health Score
  Phase 8 — RUL Prediction             → LSTM on degradation sequences
  Phase 9 — Agentic AI Orchestration   → LangGraph Agents
''')
print('=' * 68)
print(f'Dataset path: {DATA_ROOT}')
print('Replace synthetic data with real data for actual analysis results.')